# Phase 3 — Distance Matrices

Runs `src/distance_matrices/` against Phase 2's output (`reports/pattern_features.csv`) to produce: a per-species aggregate feature table, three species x species pattern-distance matrices (color/stripe/spot), and one patristic-distance matrix from the pruned molecular tree.

**No GPU needed.** Pure Python/NumPy/SciPy/Biopython, deterministic and fast - like Phase 2, this stage keeps no resumable state (a re-run just recomputes everything, cheaply). Use a **CPU runtime**.

**Prerequisite:** Phase 2 must already have produced `reports/pattern_features.csv` - run `Phase2_Pattern_Extraction.ipynb` first.

See [README.md](../README.md) (Planned Approach, step 3) and `src/distance_matrices/__init__.py` for the full design reasoning - which species are included and why, why boolean features aggregate as proportions, why `mean_spot_area` is normalized before averaging, why Euclidean distance on standardized features was chosen, and the tip-count/species-count integrity checks.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Open the same Drive-resident project Phase 1/2 used

Must resolve to the same `PROJECT_DIR` Phase 2 wrote `reports/pattern_features.csv` into.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")

if not PROJECT_DIR.exists():
    print(f"Cloning into {PROJECT_DIR} (first time - pulls ~1.8GB, be patient)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"{PROJECT_DIR} already exists - pulling latest code only.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

In [ ]:
%cd {PROJECT_DIR}

## 3. Install dependencies and add the local package to the path

`biopython` (Newick tree parsing/pruning/patristic distance) is the only dependency this phase needs beyond what Phase 2 already required.

In [ ]:
%pip install -q "biopython>=1.81"

import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import Bio, numpy, scipy
print("Deps OK - biopython", Bio.__version__, "numpy", numpy.__version__, "scipy", scipy.__version__)

## 4. Confirm Phase 2's output is actually there

In [ ]:
features_path = PROJECT_DIR / "reports" / "pattern_features.csv"
if not features_path.exists():
    print(f"Nothing at {features_path} yet - run Phase2_Pattern_Extraction.ipynb first.")
else:
    n_lines = sum(1 for _ in open(features_path, encoding="utf-8"))
    print(f"Found {features_path} ({n_lines - 1} image row(s)).")

## 5. Run the full Phase 3 pipeline

Aggregates per-species, builds all four distance matrices, and writes every output file. Restricted to the 50 species with real genetic-data phylogenetic placement (`data/phylogeny/species_coverage.csv`) - the only ones Phase 4's Kmult test can use. Default settings exclude each species' curated reference photo from its aggregate (see the package docstring for why, and the *Naso maculatus* consequence - it drops out of the analysis entirely, since its only image is the reference photo).

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s", force=True
)

from distance_matrices.config import AggregationConfig, DistanceMatrixConfig, PhylogenyConfig
from distance_matrices.pipeline import run

aggregation_config = AggregationConfig()
distance_config = DistanceMatrixConfig()
phylogeny_config = PhylogenyConfig()

result = run(aggregation_config, distance_config, phylogeny_config)
print(f"\n{len(result.species_order)} species in the primary analysis set.")

## 6. Inspect the results

Per-dimension distance summary (off-diagonal min/mean/max) as a basic sanity check - not a substitute for actually looking at the per-species feature table (`reports/species_features.csv`) and the written matrices (`outputs/*.csv`).

In [ ]:
import numpy as np

print(f"{'species':<24} n_images")
for agg in result.aggregates:
    print(f"{agg.species:<24} {agg.n_images}")

print()
for dimension, matrix in result.pattern_distance_matrices.items():
    off_diag = matrix[np.triu_indices_from(matrix, k=1)]
    print(f"{dimension:<8} distance: min={off_diag.min():.2f} mean={off_diag.mean():.2f} max={off_diag.max():.2f}")

off_diag = result.patristic_distance_matrix[np.triu_indices_from(result.patristic_distance_matrix, k=1)]
print(f"{'patristic':<8} distance: min={off_diag.min():.2f} mean={off_diag.mean():.2f} max={off_diag.max():.2f}")

## 7. (Optional) Sensitivity checks

The README's Planned Approach step 3 calls for checking whether the primary result is sensitive to the reference-image policy and to the sparse species. Both re-run the full pipeline with different settings into separate output directories, so the primary run's files aren't overwritten.

In [ ]:
# Sensitivity run 1: include reference images (recovers Naso maculatus, n should be 50 not 49)
sensitivity_config_a = AggregationConfig(
    output_csv_path=PROJECT_DIR / "reports" / "species_features_with_reference.csv",
    include_reference_images=True,
)
distance_config_a = DistanceMatrixConfig(output_dir=PROJECT_DIR / "outputs" / "sensitivity_with_reference")
phylogeny_config_a = PhylogenyConfig(
    output_path=PROJECT_DIR / "outputs" / "sensitivity_with_reference" / "patristic_distance_matrix.csv",
)
result_a = run(sensitivity_config_a, distance_config_a, phylogeny_config_a)
print(f"With reference images included: {len(result_a.species_order)} species")

# Sensitivity run 2: drop sparse species (real per-species counts range 1-22, see
# data/phylogeny/species_image_counts.csv) - min_images_per_species=5 as a starting cutoff.
sensitivity_config_b = AggregationConfig(
    output_csv_path=PROJECT_DIR / "reports" / "species_features_min5.csv",
    min_images_per_species=5,
)
distance_config_b = DistanceMatrixConfig(output_dir=PROJECT_DIR / "outputs" / "sensitivity_min5")
phylogeny_config_b = PhylogenyConfig(
    output_path=PROJECT_DIR / "outputs" / "sensitivity_min5" / "patristic_distance_matrix.csv",
)
result_b = run(sensitivity_config_b, distance_config_b, phylogeny_config_b)
print(f"Dropping species with <5 images: {len(result_b.species_order)} species")

## Next: Phase 4

The three pattern-distance matrices and the patristic-distance matrix (`outputs/*.csv`) are Phase 4's input for the secondary Mantel check; `reports/species_features.csv` (the per-species feature vectors, before reduction to distances) is what the primary Kmult test (`geomorph::physignal.z` in R) will need - see README.md's Planned Approach, step 4, for the full statistical design and why that step is planned to run in R rather than a hand-ported Python implementation.